In [ ]:
import os
import cv2
import glob
import math
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.utils import make_grid
import torchvision.transforms as T


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(42)


ROOT = r"C:\Users\Rushi\Desktop\CCDS_Split_10K"

TRAIN_DIR = os.path.join(ROOT, "train")
VAL_DIR = os.path.join(ROOT, "val")
TEST_DIR = os.path.join(ROOT, "test")

print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)
print("TEST_DIR:", TEST_DIR)

assert os.path.exists(TRAIN_DIR)
assert os.path.exists(VAL_DIR)
assert os.path.exists(TEST_DIR)

print("Dataset directories found ✔")


OUT_ROOT = r"C:\Users\Rushi\Desktop\ldm_export"

DIR_SAMPLES = os.path.join(OUT_ROOT, "samples")
DIR_CHECKPOINTS = os.path.join(OUT_ROOT, "checkpoints")
DIR_PLOTS = os.path.join(OUT_ROOT, "plots")
DIR_JSON = os.path.join(OUT_ROOT, "json")
DIR_CSV = os.path.join(OUT_ROOT, "csv")

os.makedirs(OUT_ROOT, exist_ok=True)

for d in [DIR_SAMPLES, DIR_CHECKPOINTS, DIR_PLOTS, DIR_JSON, DIR_CSV]:
    os.makedirs(d, exist_ok=True)

print("Output directories ready ✓")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def show_grid(tensor, nrow=8, title=""):
    grid = make_grid(tensor.clamp(-1, 1), nrow=nrow, normalize=False)
    grid = grid.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(10, 10))

    if grid.shape[-1] == 1:
        plt.imshow(grid[..., 0], cmap="gray")
    else:
        plt.imshow(grid)

    plt.title(title)
    plt.axis("off")
    plt.show()


In [ ]:
class OCTDataset(Dataset):
    def __init__(self, root, image_size=512, recursive=True):
        self.root = root
        self.image_size = image_size

        pattern = "**/*.png" if recursive else "*.png"
        self.paths = sorted(glob.glob(os.path.join(root, pattern), recursive=recursive))

        assert len(self.paths) > 0, f"No PNG files found in {root}"
        print(f"Found {len(self.paths)} OCT scans in: {root}")

    def __len__(self):
        return len(self.paths)

    def _load_oct_image(self, path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise RuntimeError(f"Failed to load image: {path}")
        return img

    def _pad_keep_aspect(self, img):
        h, w = img.shape
        target = self.image_size

        scale = min(target / h, target / w)
        nh, nw = int(h * scale), int(w * scale)

        img_resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)

        pad_h = target - nh
        pad_w = target - nw

        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left

        img_padded = cv2.copyMakeBorder(
            img_resized,
            top, bottom,
            left, right,
            borderType=cv2.BORDER_REFLECT_101
        )

        return img_padded

    def __getitem__(self, idx):
        path = self.paths[idx]

        img_uint8 = self._load_oct_image(path)

        img_uint8 = self._pad_keep_aspect(img_uint8)

        edges = cv2.Canny(img_uint8, 50, 150)
        edges = torch.from_numpy(edges).float().unsqueeze(0)
        edges = (edges / 255.0) * 2 - 1

        img = torch.from_numpy(img_uint8).float().unsqueeze(0)
        img = (img / 255.0) * 2 - 1

        img = img.contiguous()
        edges = edges.contiguous()

        patient = os.path.basename(os.path.dirname(path))

        return {
            "image": img,
            "cond": edges,
            "patient": patient,
            "path": path
        }


def build_loaders(train_dir, val_dir, test_dir, batch_size=4, num_workers=2, image_size=512):

    train_ds = OCTDataset(train_dir, image_size)
    val_ds   = OCTDataset(val_dir, image_size)
    test_ds  = OCTDataset(test_dir, image_size)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False
    )

    return train_loader, val_loader, test_loader


In [ ]:
import os
import glob
from collections import defaultdict
import cv2

def inspect_oct_dataset(root):
    print(f"\nInspecting: {root}")

    all_paths = sorted(glob.glob(os.path.join(root, "**/*.png"), recursive=True))
    print(f"Total PNG images: {len(all_paths)}")

    patient_counts = defaultdict(int)
    for p in all_paths:
        patient = os.path.basename(os.path.dirname(p))
        patient_counts[patient] += 1

    num_patients = len(patient_counts)
    counts = list(patient_counts.values())

    if num_patients > 0:
        print(f"Patients: {num_patients}")
        print(f"Min imgs per patient: {min(counts)}")
        print(f"Max imgs per patient: {max(counts)}")
        print(f"Avg imgs per patient: {sum(counts)/num_patients:.2f}")
    else:
        print("No patients detected")

    return num_patients, len(all_paths)


print("===== TRAIN =====")
train_patients, train_images = inspect_oct_dataset(TRAIN_DIR)

print("\n===== VAL =====")
val_patients, val_images = inspect_oct_dataset(VAL_DIR)

print("\n===== TEST =====")
test_patients, test_images = inspect_oct_dataset(TEST_DIR)


print("\n===== SUMMARY =====")
print(f"TRAIN: {train_patients} patients, {train_images} images")
print(f"VAL: {val_patients} patients, {val_images} images")
print(f"TEST: {test_patients} patients, {test_images} images")


sample_files = glob.glob(os.path.join(TRAIN_DIR, "**/*.png"), recursive=True)

if sample_files:
    img = cv2.imread(sample_files[0], cv2.IMREAD_GRAYSCALE)
    if img is not None:
        print(f"Sample resolution: {img.shape} (H,W)")
    else:
        print("Failed to load sample image")
else:
    print("No sample images found in TRAIN_DIR")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def norm_layer(ch):
    return nn.GroupNorm(min(32, ch), ch)

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.norm1 = norm_layer(ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.norm2 = norm_layer(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x):
        h = self.conv1(self.act(self.norm1(x)))
        h = self.conv2(self.act(self.norm2(h)))
        return x + h


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 4, 2, 1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class VAE(nn.Module):
    def __init__(self, im_channels=1, z_channels=32, base_ch=128):
        super().__init__()

        self.conv_in = nn.Conv2d(im_channels, base_ch, 3, padding=1)

        self.down1 = DownBlock(base_ch, base_ch * 2)
        self.res1  = ResBlock(base_ch * 2)

        self.down2 = DownBlock(base_ch * 2, base_ch * 4)
        self.res2  = ResBlock(base_ch * 4)

        self.down3 = DownBlock(base_ch * 4, base_ch * 4)
        self.res3  = ResBlock(base_ch * 4)

        mid_ch = base_ch * 4

        self.to_stats = nn.Conv2d(mid_ch, z_channels * 2, 3, padding=1)

        self.from_latent = nn.Conv2d(z_channels, mid_ch, 3, padding=1)

        self.res4 = ResBlock(mid_ch)

        self.up1 = UpBlock(mid_ch, base_ch * 4)
        self.res5 = ResBlock(base_ch * 4)

        self.up2 = UpBlock(base_ch * 4, base_ch * 2)
        self.res6 = ResBlock(base_ch * 2)

        self.up3 = UpBlock(base_ch * 2, base_ch)
        self.res7 = ResBlock(base_ch)

        self.norm_out = norm_layer(base_ch)
        self.conv_out = nn.Conv2d(base_ch, im_channels, 3, padding=1)

    def encode(self, x):
        x = self.conv_in(x)

        x = self.res1(self.down1(x))
        x = self.res2(self.down2(x))
        x = self.res3(self.down3(x))

        stats = self.to_stats(x)
        mean, logvar = torch.chunk(stats, 2, dim=1)

        logvar = torch.clamp(logvar, -10, 10)

        return mean, logvar

    def reparameterize(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mean + eps * std

    def decode(self, z):
        x = self.from_latent(z)

        x = self.res4(x)

        x = self.res5(self.up1(x))
        x = self.res6(self.up2(x))
        x = self.res7(self.up3(x))

        x = self.conv_out(F.silu(self.norm_out(x)))

        return torch.tanh(x)

    def forward(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        recon = self.decode(z)
        return recon, mean, logvar


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    vae = VAE().to(device)

    x = torch.randn(1, 1, 512, 512).to(device)
    recon, mean, logvar = vae(x)

    print("Input:", x.shape)
    print("Latent mean:", mean.shape)
    print("Reconstructed:", recon.shape)


In [ ]:
from torch.utils.data import Dataset, DataLoader
import glob
import cv2
import os
import torch
import torch.nn.functional as F


class OCTDataset(Dataset):
    def __init__(self, root, image_size=512):
        self.paths = sorted(
            glob.glob(os.path.join(root, "**/*.png"), recursive=True)
        )
        self.image_size = image_size

        if len(self.paths) == 0:
            raise RuntimeError(f"No images found in {root}")

        print(f"Loaded {len(self.paths)} images from {root}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]

        img_uint8 = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img_uint8 is None:
            raise RuntimeError(f"Failed to load image: {path}")

        img_uint8 = cv2.resize(
            img_uint8,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA
        )

        edges = cv2.Canny(img_uint8, 50, 150)
        edges = torch.from_numpy(edges).float().unsqueeze(0)
        edges = (edges / 255.0) * 2 - 1

        img = torch.from_numpy(img_uint8).float().unsqueeze(0)
        img = (img / 255.0) * 2 - 1

        return {
            "image": img,
            "cond": edges
        }


ROOT = r"C:\Users\Rushi\Desktop\CCDS_Split_10K"

TRAIN_DIR = os.path.join(ROOT, "train")
VAL_DIR   = os.path.join(ROOT, "val")
TEST_DIR  = os.path.join(ROOT, "test")


train_ds = OCTDataset(TRAIN_DIR)
val_ds   = OCTDataset(VAL_DIR)
test_ds  = OCTDataset(TEST_DIR)


BATCH_SIZE = 2

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False
)

train_loader = DataLoader(train_ds, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kwargs)

print("Conditional DataLoaders ready ✓")
print(f"Batch size: {BATCH_SIZE}")


In [ ]:
pip install pytorch-msssim

In [ ]:
!pip install lpips

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchvision.utils import make_grid, save_image
import lpips

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def norm_layer(ch):
    return nn.GroupNorm(min(32, ch), ch)


class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.norm1 = norm_layer(ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.norm2 = norm_layer(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x):
        h = self.conv1(self.act(self.norm1(x)))
        h = self.conv2(self.act(self.norm2(h)))
        return x + h


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 4, 2, 1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self, x):
        x = self.upsample(x)
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        return x


class VAE(nn.Module):

    def __init__(self, im_channels=1, z_channels=32, base_ch=128):
        super().__init__()

        self.conv_in = nn.Conv2d(im_channels, base_ch, 3, padding=1)

        self.down1 = DownBlock(base_ch, base_ch * 2)
        self.res1 = ResBlock(base_ch * 2)

        self.down2 = DownBlock(base_ch * 2, base_ch * 4)
        self.res2 = ResBlock(base_ch * 4)

        self.down3 = DownBlock(base_ch * 4, base_ch * 4)
        self.res3 = ResBlock(base_ch * 4)

        mid_ch = base_ch * 4

        self.to_stats = nn.Conv2d(mid_ch, z_channels * 2, 3, padding=1)

        self.from_latent = nn.Conv2d(z_channels, mid_ch, 3, padding=1)

        self.res4 = ResBlock(mid_ch)

        self.up1 = UpBlock(mid_ch, base_ch * 4)
        self.res5 = ResBlock(base_ch * 4)

        self.up2 = UpBlock(base_ch * 4, base_ch * 2)
        self.res6 = ResBlock(base_ch * 2)

        self.up3 = UpBlock(base_ch * 2, base_ch)
        self.res7 = ResBlock(base_ch)

        self.norm_out = norm_layer(base_ch)
        self.conv_out = nn.Conv2d(base_ch, im_channels, 3, padding=1)

    def encode(self, x):
        x = self.conv_in(x)
        x = self.res1(self.down1(x))
        x = self.res2(self.down2(x))
        x = self.res3(self.down3(x))
        stats = self.to_stats(x)
        mean, logvar = torch.chunk(stats, 2, dim=1)
        logvar = torch.clamp(logvar, -10, 10)
        return mean, logvar

    def reparameterize(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mean + eps * std

    def decode(self, z):
        x = self.from_latent(z)
        x = self.res4(x)
        x = self.res5(self.up1(x))
        x = self.res6(self.up2(x))
        x = self.res7(self.up3(x))
        x = self.conv_out(F.silu(self.norm_out(x)))
        return torch.tanh(x)

    def forward(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        recon = self.decode(z)
        return recon, mean, logvar


lpips_loss = lpips.LPIPS(net='vgg').to(device)
lpips_loss.eval()

for p in lpips_loss.parameters():
    p.requires_grad = False


def perceptual_loss(pred, target):
    return lpips_loss(
        pred.repeat(1, 3, 1, 1),
        target.repeat(1, 3, 1, 1)
    ).mean()


def kl_loss(mean, logvar):
    logvar = torch.clamp(logvar, -30, 20)
    return -0.5 * torch.sum(
        1 + logvar - mean.pow(2) - logvar.exp(),
        dim=[1, 2, 3]
    ).mean()


def recon_loss(pred, target):
    return F.l1_loss(pred, target)


def train_vae(
    vae,
    train_loader,
    val_loader,
    epochs=150,
    lr=5e-5,
    out_dir="vae_outputs"
):

    os.makedirs(out_dir, exist_ok=True)
    preview_dir = os.path.join(out_dir, "previews")
    os.makedirs(preview_dir, exist_ok=True)

    optimizer = torch.optim.AdamW(vae.parameters(), lr=lr)

    scaler = torch.amp.GradScaler(
        enabled=(device.type == "cuda")
    )

    preview = next(iter(val_loader))["image"][:4].to(device)

    best_val = float("inf")

    for epoch in range(1, epochs + 1):

        beta = min(1e-4, epoch * 5e-6)
        perc_weight = 0.005

        vae.train()
        total_recon = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):

            img = batch["image"].to(device)

            optimizer.zero_grad()

            with torch.amp.autocast(enabled=(device.type == "cuda")):

                recon, mean, logvar = vae(img)

                loss_l1 = recon_loss(recon, img)
                loss_p = perceptual_loss(recon, img)
                loss_k = kl_loss(mean, logvar)

                loss = loss_l1 + perc_weight * loss_p + beta * loss_k

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            total_recon += loss_l1.item()

        total_recon /= len(train_loader)

        vae.eval()
        val_recon = 0

        with torch.no_grad():
            for batch in val_loader:
                img = batch["image"].to(device)
                recon, _, _ = vae(img)
                val_recon += recon_loss(recon, img).item()

        val_recon /= len(val_loader)

        print(f"\nEpoch {epoch}")
        print(f"Train L1: {total_recon:.4f} | Val L1: {val_recon:.4f}")

        with torch.no_grad():
            recon, _, _ = vae(preview)

        vis = torch.cat([preview, recon], dim=0)
        vis = torch.clamp((vis + 1) / 2, 0, 1)

        save_image(
            vis,
            os.path.join(preview_dir, f"epoch_{epoch}.png"),
            nrow=4
        )

        if val_recon < best_val:
            best_val = val_recon
            torch.save(
                vae.state_dict(),
                os.path.join(out_dir, "vae_best.pth")
            )
            print("New best model saved")

    print("VAE Training Complete")


vae = VAE(
    z_channels=32,
    base_ch=128
).to(device)

train_vae(
    vae,
    train_loader,
    val_loader
)


In [ ]:
# ============================================================
# OCT CONDITIONAL LATENT DIFFUSION (LOCAL VERSION)
# z = 32 , base = 128
# EMA + CHECKPOINT + RESUME + ATTENTION + COSINE SCHEDULE
# ============================================================

import os
import glob
import cv2
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
import lpips
from torchmetrics.image import StructuralSimilarityIndexMeasure

# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# LOCAL PATHS (CHANGE ONLY THIS SECTION)
# ============================================================

ROOT = r"D:\Rushi_OCT_Diffusion\CCDS_Split_10K"

TRAIN_PATH = os.path.join(ROOT,"train")
VAL_PATH   = os.path.join(ROOT,"val")

VAE_PATH = r"D:\Rushi_OCT_Diffusion\models\VAE_CONDITIONAL_LDM.pth"

OUT_DIR = r"D:\Rushi_OCT_Diffusion\Conditional_LDM"

PREVIEW_DIR = os.path.join(OUT_DIR,"previews")
CHECKPOINT_PATH = os.path.join(OUT_DIR,"ldm_checkpoint.pth")
BEST_MODEL_PATH = os.path.join(OUT_DIR,"ldm_best_model.pth")

os.makedirs(PREVIEW_DIR,exist_ok=True)

# ============================================================
# DATASET
# ============================================================

class OCTDataset(Dataset):

    def __init__(self,root,size=512):

        self.paths = sorted(
            glob.glob(os.path.join(root,"**/*.png"),recursive=True)
        )

        self.size = size

        print(f"{len(self.paths)} images loaded from {root}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self,i):

        img = cv2.imread(self.paths[i],cv2.IMREAD_GRAYSCALE)

        img = cv2.resize(img,(self.size,self.size))

        img = torch.from_numpy(img).float()/255.0
        img = img.unsqueeze(0)*2 - 1

        return {"image":img}

train_loader = DataLoader(
    OCTDataset(TRAIN_PATH),
    batch_size=4,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    OCTDataset(VAL_PATH),
    batch_size=4,
    shuffle=False,
    num_workers=0
)

# ============================================================
# VAE ARCHITECTURE
# ============================================================

def norm_layer(ch):
    return nn.GroupNorm(min(32,ch),ch)

class ResBlock(nn.Module):

    def __init__(self,ch):

        super().__init__()

        self.norm1 = norm_layer(ch)
        self.conv1 = nn.Conv2d(ch,ch,3,padding=1)

        self.norm2 = norm_layer(ch)
        self.conv2 = nn.Conv2d(ch,ch,3,padding=1)

        self.act = nn.SiLU()

    def forward(self,x):

        h = self.conv1(self.act(self.norm1(x)))
        h = self.conv2(self.act(self.norm2(h)))

        return x + h


class DownBlock(nn.Module):

    def __init__(self,in_ch,out_ch):

        super().__init__()

        self.conv = nn.Conv2d(in_ch,out_ch,4,2,1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self,x):

        return self.act(self.norm(self.conv(x)))


class UpBlock(nn.Module):

    def __init__(self,in_ch,out_ch):

        super().__init__()

        self.conv = nn.ConvTranspose2d(in_ch,out_ch,4,2,1)
        self.norm = norm_layer(out_ch)
        self.act = nn.SiLU()

    def forward(self,x):

        return self.act(self.norm(self.conv(x)))


class VAE(nn.Module):

    def __init__(self):

        super().__init__()

        b = 128
        z = 32

        self.conv_in = nn.Conv2d(1,b,3,1,1)

        self.down1 = DownBlock(b,b*2)
        self.res1 = ResBlock(b*2)

        self.down2 = DownBlock(b*2,b*4)
        self.res2 = ResBlock(b*4)

        self.down3 = DownBlock(b*4,b*4)
        self.res3 = ResBlock(b*4)

        self.to_stats = nn.Conv2d(b*4,z*2,3,1,1)

        self.from_latent = nn.Conv2d(z,b*4,3,1,1)

        self.res4 = ResBlock(b*4)

        self.up1 = UpBlock(b*4,b*4)
        self.res5 = ResBlock(b*4)

        self.up2 = UpBlock(b*4,b*2)
        self.res6 = ResBlock(b*2)

        self.up3 = UpBlock(b*2,b)
        self.res7 = ResBlock(b)

        self.norm_out = norm_layer(b)
        self.conv_out = nn.Conv2d(b,1,3,1,1)

    def encode(self,x):

        x = self.conv_in(x)

        x = self.res1(self.down1(x))
        x = self.res2(self.down2(x))
        x = self.res3(self.down3(x))

        mean,logvar = torch.chunk(self.to_stats(x),2,1)

        return mean,logvar.clamp(-10,10)

    def decode(self,z):

        x = self.from_latent(z)

        x = self.res4(x)

        x = self.res5(self.up1(x))
        x = self.res6(self.up2(x))
        x = self.res7(self.up3(x))

        return torch.tanh(self.conv_out(F.silu(self.norm_out(x))))

# ============================================================
# LOAD VAE
# ============================================================

vae = VAE().to(device)

vae.load_state_dict(torch.load(VAE_PATH,map_location=device))

vae.eval()
vae.requires_grad_(False)

print("VAE loaded")

# ============================================================
# CONDITION (EDGE MAP)
# ============================================================

sobel_x = torch.tensor(
    [[1,0,-1],[2,0,-2],[1,0,-1]],
    device=device
).view(1,1,3,3).float()

sobel_y = torch.tensor(
    [[1,2,1],[0,0,0],[-1,-2,-1]],
    device=device
).view(1,1,3,3).float()

def get_condition(img):

    img = (img+1)/2

    img = F.avg_pool2d(img,3,stride=1,padding=1)

    gx = F.conv2d(img,sobel_x,padding=1)
    gy = F.conv2d(img,sobel_y,padding=1)

    edges = torch.sqrt(gx**2 + gy**2)

    edges = edges/(edges.amax(dim=[1,2,3],keepdim=True)+1e-8)

    edges = edges*2 - 1

    return F.interpolate(edges,size=(64,64))

# ============================================================
# UNET
# ============================================================

unet = UNet2DModel(
    sample_size=64,
    in_channels=33,
    out_channels=32,
    layers_per_block=2,
    block_out_channels=(128,256,512,512),
    down_block_types=(
        "DownBlock2D",
        "AttnDownBlock2D",
        "AttnDownBlock2D",
        "DownBlock2D"
    ),
    up_block_types=(
        "UpBlock2D",
        "AttnUpBlock2D",
        "AttnUpBlock2D",
        "UpBlock2D"
    )
).to(device)

ema_unet = copy.deepcopy(unet)

for p in ema_unet.parameters():
    p.requires_grad_(False)

ema_decay = 0.999

def update_ema():

    with torch.no_grad():

        for e,p in zip(ema_unet.parameters(),unet.parameters()):
            e.data.mul_(ema_decay).add_(p.data,alpha=1-ema_decay)

# ============================================================
# DIFFUSION
# ============================================================

scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2"
)

ddim = DDIMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2"
)

optimizer = torch.optim.AdamW(unet.parameters(),lr=1e-4)

# ============================================================
# METRICS
# ============================================================

lpips_metric = lpips.LPIPS(net="vgg").to(device).eval()
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

# ============================================================
# GENERATE
# ============================================================

@torch.no_grad()
def generate(img):

    ema_unet.eval()

    mean,_ = vae.encode(img)

    cond = get_condition(img)

    z = torch.randn_like(mean)

    ddim.set_timesteps(100)

    for t in ddim.timesteps:

        xt_cond = torch.cat([z,cond],dim=1)

        noise_pred = ema_unet(xt_cond,t).sample

        z = ddim.step(noise_pred,t,z).prev_sample

    return vae.decode(z)